In [11]:
import pandas as pd
from typing import Literal

In [12]:
def log_step(
    df: pd.DataFrame,
    step_name: str | None = None,
    variant: Literal["info", "describe", "memory", "head", "verbose"] = "info",
) -> pd.DataFrame:
    """Print a dataframe during a chained method"""
    if step_name:
        print(f"-----------------------{step_name}-----------------------")

    match variant:
        case "info":
            print(df.info())
        case "describe":
            print(df.describe())
        case "memory":
            print(df.memory_usage(deep=True))
        case "head":
            print(df.head())
        case "verbose":
            print(df)
    return df

In [27]:
sales_df = pd.read_csv(r".\data\Sales.csv")
customer_df = pd.read_csv(r".\data\Customers.csv")
product_df = pd.read_csv(r".\data\Products.csv")

# Merge dataframes together
merged_df = (
    sales_df.copy()
    # .pipe(log_step, "Original Sales Data")
    .merge(customer_df, how="left", on="CustomerKey")
    # .pipe(log_step, "Merged Customers")
    .merge(product_df, how="left", on="ProductKey")
    # .pipe(log_step, "Merged Products")
    # .query("UnitPrice > 500 and UnitPrice <= 1000")
    .filter(
        items=[
            # "ProductKey",
            # "CustomerKey",
            # "EnglishProductName",
            "UnitPrice",
            # "OrderQuantity",
            "OrderDate",
            "SalesAmount",
        ]
    )
    # .set_index('OrderDate').resample('W').size()
    # .reset_index(name='count')
    # .pipe(log_step, "Query Result", "verbose")
)

# Convert date to datetime
merged_df["OrderDate"] = pd.to_datetime(
    merged_df["OrderDate"], format="%Y-%m-%d %H:%M:%S.%f"
)

# Set index to datetime, and group by week
merged_df = (
    merged_df.set_index("OrderDate").resample("W")["SalesAmount"].sum().reset_index()
)

print(merged_df)

# Chain methods together
# clean_sales_df = (
#     sales_df
#     .copy()
#     .query('SalesAmount > 3000')
#     .pipe(log_step, "Sales Amount > 3000")
#     .reset_index()
# )
# print("aksdflakjdsf\n\n\n\n")
# print(clean_sales_df[['ProductKey', 'SalesAmount']])

     OrderDate  SalesAmount
0   2011-01-02   65589.7546
1   2011-01-09   89938.5464
2   2011-01-16  104404.9064
3   2011-01-23  118525.6846
4   2011-01-30  123555.4310
..         ...          ...
157 2014-01-05   11403.0900
158 2014-01-12    9996.3100
159 2014-01-19   11774.2100
160 2014-01-26   11711.0100
161 2014-02-02    4121.2200

[162 rows x 2 columns]


In [ ]:
from bokeh.models import ColumnDataSource
from bokeh.plotting import figure, show
from bokeh.io import output_notebook

# Convert dataframe for bokeh plotting
merged_source = ColumnDataSource(merged_df)

print(merged_source)

p = figure(
    title="Sales per Week",
    x_axis_type="datetime",
    x_axis_label="Years Active",
    y_axis_label="Value ($)",
    width=600,
    height=400,
)

p.vbar(
    x="OrderDate",
    top="SalesAmount",
    source=merged_source,
    line_width=2,
    color="navy",
    line_color="green",
)

output_notebook()
show(p)

ColumnDataSource(id='p1475', ...)


Loading BokehJS ...